<a href="https://colab.research.google.com/github/loukikjoshi06-ops/SIH-26016/blob/main/AI_Model/notebooks/01_qwen2_5_vl_7b_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Sat Sep 26 04:02:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes peft qwen-vl-utils trl torchvision


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 71.2 MB/s eta 0:00:00


In [3]:
!pip install -q transformers bitsandbytes accelerate peft trl qwen-vl-utils paddleocr pdf2image
!apt-get -qq install -y poppler-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295

In [4]:
from google.colab import files
uploaded = files.upload()  # click the button and pick your PDF

Saving 7-12-sample-of-Maharashtra.pdf to 7-12-sample-of-Maharashtra.pdf


In [15]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

# 4-bit NF4 Quantization (keeps VRAM around ~6-7 GB on your 15GB T4 GPU)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

# Resolution bounds to prevent VRAM spikes
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28
)

print("⏳ Downloading and loading Qwen2.5-VL-7B in 4-bit (takes ~2 minutes)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=compute_dtype,
    quantization_config=bnb_config
)

print("✅ Qwen2.5-VL-7B loaded successfully into Colab GPU!")

⏳ Downloading and loading Qwen2.5-VL-7B in 4-bit (takes ~2 minutes)...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✅ Qwen2.5-VL-7B loaded successfully into Colab GPU!


In [11]:
!pip install -q peft trl

cell 4

In [12]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

Mounted at /content/drive
✅ Drive mounted!


In [ ]:
model.save_pretrained("/content/drive/MyDrive/marathi-ocr-model/lora_adapter")
processor.save_pretrained("/content/drive/MyDrive/marathi-ocr-model/lora_adapter")
print("✅ Model saved!")

✅ Model saved!


In [ ]:
!pip install -q transformers bitsandbytes accelerate peft qwen-vl-utils pdf2image
!apt-get -qq install -y poppler-utils

from google.colab import drive
drive.mount('/content/drive')
print("✅ Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 70.5 MB/s eta 0:00:00
Selecting previously unselected package poppler-utils.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
Mounted at /content/drive
✅ Ready!


loading saved (LoRA) weights
|

In [13]:
from google.colab import files
from pdf2image import convert_from_path

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

# Convert PDF pages into images at 300 DPI
images = convert_from_path(pdf_filename, dpi=300)
print(f"✅ Document loaded! Total pages: {len(images)}")

Saving 7-12-sample-of-Maharashtra.pdf to 7-12-sample-of-Maharashtra (1).pdf
✅ Document loaded! Total pages: 2


In [21]:
import json, re, torch
from qwen_vl_utils import process_vision_info

# =========================================================================
# 1. 100% RELIABLE MARATHI -> ENGLISH DIGIT CONVERTER
# =========================================================================
def marathi_to_english_digits(text):
    if not isinstance(text, str):
        return text
    marathi_digits = "०१२३४५६७८९"
    english_digits = "0123456789"
    return text.translate(str.maketrans(marathi_digits, english_digits))

def standardize_json(data):
    if isinstance(data, dict):
        return {k: standardize_json(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [standardize_json(item) for item in data]
    elif isinstance(data, str):
        return marathi_to_english_digits(data)
    return data

# =========================================================================
# 2. USE PRISTINE 300 DPI IMAGE (No destructive contrast filter)
# =========================================================================
raw_img = images[0].convert("RGB")

# =========================================================================
# 3. ANCHOR PROMPT (Exact columns, prevents misreading headers as names)
# =========================================================================
SYSTEM_PROMPT = """You are an expert reading a Maharashtra 7/12 (सातबारा उतारा) land record.
Extract all details into a JSON object.

EXTRACTION INSTRUCTIONS:
1. HEADER DETAILS (Top section):
   - "district": Look at header top right "जिल्हा :- ..." (e.g. पुणे)
   - "taluka": Look at header top middle "तालुका :- ..." (e.g. मावळ)
   - "village": Look at header top left "गाव :- ...". The village name is the word directly before the code in parentheses. WARNING: "गाव नमुना सात" is the title of the form, NEVER use that as village name!
   - "village_code": The 6-digit number inside brackets next to the village (e.g. ५५६०२३)

2. TABLE DETAILS:
   - "survey_number": The large number under "गट क्रमांक व उपविभाग" (e.g. ३९२)
   - "khata_number": Number in the "खाते क्र." column (e.g. १०३)
   - "occupancy_class": Text under "भूधारणा पद्धती" (e.g. भोगवटदार वर्ग - १)
   - "owners": List of names in the "भोगवटदाराचे नाव" column. Ignore table lines like 'सामाईक क्षेत्र'. Do NOT include words from other columns like 'कुळाचे नाव व खंड'.
   - "area":
       - "unit": "हे.आर.चौ.मी"
       - "total": Value under "एकूण क्षेत्र (अ+ब)" (e.g. १.१६.३०)
       - "cultivable": Value under "एकूण ला.यो. क्षेत्र" (e.g. ०.३१.८०)
       - "uncultivable": Value under "एकूण पो.ख.क्षेत्र" (e.g. ०.८४.५०)
   - "assessment": Value under "आकारणी" (e.g. १.३७)
   - "encumbrances": Any bank bojas or pending mutations under "इतर अधिकार". If empty or 'नाही', return null.
   - "last_mutation_number": The approved mutation number under "शेवटचा फेरफार क्रमांक" (e.g. ५४२३).

Output ONLY this raw JSON schema in Marathi Devanagari numerals:
{
  "district": null,
  "taluka": null,
  "village": null,
  "village_code": null,
  "survey_number": null,
  "khata_number": null,
  "occupancy_class": null,
  "owners": [],
  "area": {
    "unit": "हे.आर.चौ.मी",
    "total": null,
    "cultivable": null,
    "uncultivable": null
  },
  "assessment": null,
  "encumbrances": null,
  "last_mutation_number": null
}"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": raw_img},
            {"type": "text", "text": "Extract all data into the requested JSON schema."}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
vision_imgs, _ = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=vision_imgs,
    return_tensors="pt",
    padding=True,
    truncation=False
).to(model.device)

print("⏳ Running 7B Model Inference on Pristine Image...")
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=768,
        temperature=0.01
    )

# =========================================================================
# 4. JSON PARSER & AUTO-REPAIR
# =========================================================================
raw_output = processor.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
clean_text = raw_output.replace("```json", "").replace("```", "").strip()

match = re.search(r'\{.*\}', clean_text, re.DOTALL)
json_str = match.group(0) if match else clean_text

# Fix missing / trailing commas
json_str = re.sub(r'([0-9"a-zA-Z\u0900-\u097F])\s*\n\s*(")', r'\1,\n\2', json_str)
json_str = re.sub(r',\s*([\}\]])', r'\1', json_str)

try:
    extracted_json = json.loads(json_str)
except Exception:
    print("⚠️ Raw output fallback:")
    print(raw_output)
    extracted_json = {"raw": raw_output}

# =========================================================================
# 5. STANDARDIZE NUMBERS TO CLEAN ENGLISH DIGITS
# =========================================================================
final_production_json = standardize_json(extracted_json)

print("\n📄 Restored High-Accuracy 7/12 JSON:")
print(json.dumps(final_production_json, indent=2, ensure_ascii=False))

⏳ Running 7B Model Inference on Pristine Image...

📄 Restored High-Accuracy 7/12 JSON:
{
  "district": "पुणे",
  "taluka": "मावळ",
  "village": "गाव नमुना सात",
  "village_code": "556023",
  "survey_number": "392",
  "khata_number": "103",
  "occupancy_class": "भोगवटदार वर्ग - 1",
  "owners": [
    "मोहनीश रमेश चाबला"
  ],
  "area": {
    "unit": "हे.आर.चौ.मी",
    "total": "1.16.30",
    "cultivable": "0.31.80",
    "uncultivable": "0.84.50"
  },
  "assessment": "1.37",
  "encumbrances": null,
  "last_mutation_number": "5423"
}
